Installing Dependencies!

In [71]:
!pip install -qU langchain-community pypdf

In [72]:
!pip install -qU langchain langchain-huggingface sentence_transformers

In [73]:
from langchain_community.document_loaders import PyPDFLoader

Loading Data


In [74]:
file_path = "/content/resume.pdf"
loader = PyPDFLoader(file_path)
doc = loader.load()

Splitting data


In [75]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200,
    add_start_index = True  #Track index in original document
)

all_splits = text_splitter.split_documents(doc)
print(all_splits)


[Document(metadata={'producer': 'pdfTeX-1.40.29', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-10-07T08:31:28+00:00', 'author': '', 'keywords': '', 'moddate': '2026-10-07T08:31:28+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.29 (TeX Live 2026) kpathsea version 6.4.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/content/resume.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'start_index': 0}, page_content='HRUDAY KRISHNA REDDY BANKA\n+91-9959668335 | hrudaykrishnar03@gmail.com | linkedin.com/in/Hruday | github.com/hrudaykrishnareddy03\nMachine Learning•Generative AI•Large Language Models (LLMs)•AI Agents•Prompt\nEngineering•Python\nSUMMARY\nFinal-year B.Tech Computer Science student graduating May 2027 with hands-on experience in Machine Learning,\nGenerative AI, LLM-based applications, and data analysis. Built ML prediction systems, AI-powered applications, and\na LangChain-based AI agent using Python, Scikit-learn, Google G

Initializing Embedding Model

In [76]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-mpnet-base-v2"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Storing Embeddings in Vector Database

In [77]:
!pip install -qU langchain-chroma

In [78]:
from langchain_chroma import Chroma


In [79]:
vector_store = Chroma(
    collection_name="Research-collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db"
)

document_ids = vector_store.add_documents(documents=all_splits),
sample = vector_store.get(limit=1, include=["embeddings", "documents"])
print(f"Embedding dimensions: {len(sample['embeddings'][0])}")
print(sample)
print(document_ids[:3])

Embedding dimensions: 768
{'ids': ['85ccb625-c5f0-4e71-b196-79dbb9d7e055'], 'embeddings': array([[ 5.30871339e-02,  4.00415771e-02, -4.24599387e-02,
         2.88901990e-03, -1.25898262e-02,  6.91890763e-03,
         8.33183806e-03, -7.64228450e-03,  5.59619591e-02,
        -5.33811301e-02,  4.57341820e-02,  5.56809595e-03,
        -2.96985451e-02,  1.17781557e-01,  3.44325006e-02,
        -2.73284167e-02,  2.28371602e-02, -1.24469651e-02,
        -1.67362785e-04, -9.59320087e-03, -3.04876007e-02,
         2.74193175e-02, -1.74496211e-02,  5.17499968e-02,
        -4.33577783e-02, -3.58490236e-02, -3.40747423e-02,
         1.43008139e-02, -2.55590100e-02, -9.30278096e-03,
         5.93614997e-03, -2.36433170e-05,  1.41858896e-02,
         4.85801250e-02,  2.62123763e-06, -8.31768438e-02,
        -3.97966104e-03,  1.14387390e-03, -2.08397079e-02,
        -3.24995928e-02,  4.10843194e-02, -8.43016431e-03,
         3.22823122e-04,  3.86739224e-02, -6.24362938e-02,
         2.27085873e-02, 

Custom Tool creation for agent

In [80]:
from langchain_core.tools import tool

In [81]:
@tool
def retrieve_context(query: str) -> str:
  """Retrieve information from the given pdf."""
  retrieved_docs = vector_score.similarity_search(query, k=k)

  docs_content = ""
  for doc in retrieved_docs:
      docs_content += f"Source: {doc.metadata}\n"
      docs_content += f"Content: {doc.page_content}\n\n"

  return docs_content, retrieved_docs


Using Tavily Search for acknoledgement of latest information

In [82]:
!pip install -qU langchain-tavily

In [83]:
tavily_api_key = userdata.get('TAVILY_API_KEY')

web_search_tool = TavilySearch(
    max_results=3,
    search_depth="advanced",
    tavily_api_key=tavily_api_key
)

Initializing Model

In [84]:
!pip install -qU langchain-google-genai

In [85]:
from langchain.chat_models import init_chat_model
from google.colab import userdata
from langchain.tools import tool
from langchain_tavily import TavilySearch

google_api_key = userdata.get('GEMINI_API_KEY')

model = init_chat_model(
    "google_genai:gemini-3.5-flash",
    api_key=google_api_key
)


Creating Agent

In [86]:
from langchain.agents import create_agent

In [87]:
system_prompt = """You are a helpful research assistant with access to two tools:

1. retrieve_from_pdf: Use this to find information from the
   "Attention Is All You Need" research paper

2. TavilySearch: Use this to find current information
   not in the paper (recent events, updates, etc.)

Strategy:
- For questions about the paper content → use retrieve_from_pdf
- For questions about recent events or topics not in the paper → use TavilySearch
"""

In [88]:
agent = create_agent(
    model=model,
    tools=[retrieve_context, web_search_tool],
    system_prompt= system_prompt
)

In [89]:
user_query = "Please give suggestions, how to improve my resume to become AI Engineer"

In [90]:
response = agent.invoke({
    "messages": [{"role": "user", "content": user_query}]
    })

In [91]:
from pprint import pprint

In [92]:
pprint(response["messages"][-1].content)

[{'extras': {'signature': 'EosHCogHAWkUfROCIDBLg74Ac+8o6Dl5rT/taFEA4ovENoSI8N1iUrbwkSsU+CvKGp4NBJSHq9GR9yYuUI1Ceitt6FWJ02H4jSY091BFgEYRQMfQZESm6qiIqGBtHXEGpgHOA6h5S1p4GDxeDpunltrvgx3SNA2UwEHQF414KuYHs9zEP+BkST10e100KjVYB2LuTgUxA5olb5oIySgLEOhX3qrmsnxqjSG6edoOsy4be129Mgi2WKn5uWgoRzNiHCLqqHpPSV14tqfx7t7CylTILsWyQ73X52TfePh1bzH8K+ZX/fhXJWjkrU/oMWEiJDq7NOiQfeFVg+Rixr/FipSvzn5MlLERTpePA5zd4PVyco0Eb/uM6u8sEBbOLf6mtTeBzioefsT/Ptdkg7eRtzZRODBKrzO6R8ExG3MNM3UvtDdzD5KtwCajMEd+YC7U4kmDrF1POy8TelKwQ4/N/EexKuX0YRF0AGn/iSf8YWxdRCteDAynBvgrVPYObjC14GjS1gEdFNX2XJKVYi3yvbx9Agq5ai6ho/YIVzJqX8yYH5hPZYncW6smaiM1mAhvlvsWgzNfED+rtWYcp7XdGDI3oTWihjSwpEDhw9yow3fHCy4hrQW3bdD/IrakgHp2pqpE3kF3WrJWkUTohua7r5R1K9JZmX9xoRxB2mZNz+phex8eNmulm7qmJktASFjaRhqG19LEBZI1UontOVxQSxQ/44oSUNbULaEUrFOB6hNvcRgpCOxSYW6jkwOo3LMUs18AIA5G9zkdNDEqkf5Z5hPHQl3wReIStdJVZJYIkyhttbvivet6J5ir3CoLtPz7AHnIG7Zg4wlR73j5u6jQ+Zb9QrIP+F6yPca1K969cUMvmcpGEr2uKdUMN3E36Cs8sVITCndJq/pwiOV8avJYixlhAoAi0JwCfdETn7RdlWt7cozRXNfgfJ/mzHbQbEIwx2ZMAJrwj2MT4